In [1]:
# Imports & Path Configuration
import os
import sys
import warnings
from collections import Counter
import nltk
from google.genai import types
import uvicorn
from dotenv import load_dotenv

# --- PATH CONFIGURATION ---
# In Jupyter, __file__ is not defined. We use os.getcwd() to find the location.
# This assumes the notebook is running inside the 'agents/' folder.
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
predictors_dir = os.path.join(parent_dir, 'pred_models_training')

# Add to sys.path so Python can find predictors.py
if predictors_dir not in sys.path:
    sys.path.append(predictors_dir)
    print(f"Added to sys.path: {predictors_dir}")

# --- PREDICTORS API IMPORT ---
try:
    from predictors import (
        predict_news_coverage,
        predict_intent,
        predict_sensationalism,
        predict_article_stance,
        analyze_complete_article
    )
    print(f"✅ Successfully imported predictors from {predictors_dir}")
except ImportError as e:
    print(f"❌ Failed to import predictors: {e}")
    print(f"   Current sys.path: {sys.path}")
    # We raise error here because the agents won't work without this
    raise e

# --- ADK IMPORTS ---
from google.adk.agents import Agent
from google.adk.tools import AgentTool, google_search
from google.adk.models.google_llm import Gemini
from google.adk.a2a.utils.agent_to_a2a import to_a2a

warnings.filterwarnings("ignore")

# Ensure NLTK resources
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

print("Imports complete.")

Added to sys.path: /home/yal149/private/DSC180A-GroupNull/pred_models_training
✅ Successfully imported predictors from /home/yal149/private/DSC180A-GroupNull/pred_models_training
Imports complete.


In [2]:
# Tool Wrappers

def get_sentences(text: str):
    """Helper to split text into sentences for granular analysis."""
    try:
        sentences = nltk.sent_tokenize(text or "")
        # Filter out very short fragments
        sentences = [s.strip() for s in sentences if len(s.strip()) > 10]
        if not sentences and text.strip():
            sentences = [text.strip()]
        return sentences if sentences else []
    except Exception:
        return [text.strip()] if text else []

# --- TOOL WRAPPERS ---

def tool_news_topic(article_text: str) -> dict:
    print("\n   [⚙️ MODEL EXECUTING] 🟢 'tool_news_topic' running...")
    sentences = get_sentences(article_text)
    votes = []
    for s in sentences:
        try:
            res = predict_news_coverage(s)
            # FIX: Ignore missing data labels ("None", "nan")
            if res and res.get("label") and str(res["label"]) not in ["None", "nan", "unknown"]:
                votes.append(res["label"])
        except Exception as e:
            print(f"      ❌ tool_news_topic error: {e}")
            continue
            
    topic = Counter(votes).most_common(1)[0][0] if votes else "unknown"
    
    result = {"news_coverage": topic}
    print(f"   [✅ MODEL OUTPUT] 🟢 'tool_news_topic' returned: {result}")
    return result

def tool_intent(article_text: str) -> dict:
    print("\n   [⚙️ MODEL EXECUTING] 🟢 'tool_intent' running...")
    sentences = get_sentences(article_text)
    votes = []
    for s in sentences:
        try:
            res = predict_intent(title="", body=s)
            if res and res.get("label"):
                votes.append(res["label"])
        except Exception as e:
            # FIX: Stop failing silently! Print the actual error.
            print(f"      ❌ tool_intent error: {e}")
            continue
            
    intent = Counter(votes).most_common(1)[0][0] if votes else "unknown"
    
    result = {"intent": intent}
    print(f"   [✅ MODEL OUTPUT] 🟢 'tool_intent' returned: {result}")
    return result

def tool_sensationalism(article_text: str) -> dict:
    print("\n   [⚙️ MODEL EXECUTING] 🟢 'tool_sensationalism' running...")
    sentences = get_sentences(article_text)
    votes = []
    for s in sentences:
        try:
            res = predict_sensationalism(statement=s)
            if res and res.get("label"):
                votes.append(str(res["label"]))
        except Exception as e:
            print(f"      ❌ tool_sensationalism error: {e}")
            continue
            
    final_label = Counter(votes).most_common(1)[0][0] if votes else "neutral"
    
    result = {"sensationalism": final_label}
    print(f"   [✅ MODEL OUTPUT] 🟢 'tool_sensationalism' returned: {result}")
    return result

def tool_stance(article_text: str) -> dict:
    print("\n   [⚙️ MODEL EXECUTING] 🟢 'tool_stance' running...")
    try:
        res = predict_article_stance(article_text=article_text)
        label = res.get("label", "neutral")
    except Exception as e:
        print(f"      ❌ tool_stance error: {e}")
        label = "neutral"
        
    result = {"stance": label}
    print(f"   [✅ MODEL OUTPUT] 🟢 'tool_stance' returned: {result}")
    return result
        
print("Tool wrappers defined.")

Tool wrappers defined.


In [3]:
# Load the .env file
load_dotenv() 

# Retrieve the key
api_key = os.getenv("GEMINI_API_KEY")

# Check if it loaded correctly
if not api_key:
    raise ValueError("❌ GOOGLE_API_KEY not found! Make sure you created the .env file.")

In [4]:
# Agent Definitions

retry_config = types.HttpRetryOptions(
    attempts=5,
    exp_base=7,
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],
)

# 1. Sensationalism Agent
sensationalism_agent = Agent(
    name="Sensationalism_Analyst",
    model=Gemini(model="gemini-3-flash-preview", retry_options=retry_config),
    instruction=(
        "Analyze the provided article for sensationalism."
        "Output your final verdict ('sensational' or 'neutral')."
    )
)

# 2. Stance Agent
stance_agent = Agent(
    name="Stance_Analyst",
    model=Gemini(model="gemini-3-flash-preview", retry_options=retry_config),
    instruction=(
        "Determine the author's stance in the provided article."
        "Output your final verdict ('support', 'deny', or 'neutral')."
    )
)

# 3. Context Veracity Agent
context_agent = Agent(
    name="Context_Veracity_Analyst",
    model=Gemini(model="gemini-3-flash-preview"),
    description="A specialized agent for verifying the contextual veracity of news articles.",
    instruction=(
        "Evaluate the factual accuracy and contextual coherence of the provided article."
        "Output your final verdict ('Accurate' or 'Inaccurate')."
    )
)

# 4. News Coverage Agent
news_coverage_agent = Agent(
    name="News_Coverage_Analyst",
    model=Gemini(model="gemini-3-flash-preview", retry_options=retry_config),
    instruction=(
        "Categorize the primary topic of the provided news article."
        "Output your final topic label as the verdict."
    )
)

# 5. Intent Agent
intent_agent = Agent(
    name="Intent_Analyst",
    model=Gemini(model="gemini-3-flash-preview", retry_options=retry_config),
    instruction=(
        "Identify the author's primary intent in writing the provided article."
        "Output your final verdict ('Inform', 'Persuade', 'Entertain', or 'Deceive')."
    )
)

# 6. Title vs Body Agent
title_body_agent = Agent(
    name="Title_Body_Analyst",
    model=Gemini(model="gemini-3-flash-preview", retry_options=retry_config),
    instruction=(
        "Determine the relationship between the article's headline and its body text."
        "Output your final verdict ('Agree', 'Discuss', 'Contradicts', or 'Unrelated')."
    )
)

print("Agents initialized.")

Agents initialized.


In [5]:
# Root Coordinator

root_agent = Agent(
    name="ParallelCoordinator",
    model=Gemini(model="gemini-3-flash-preview", retry_options=retry_config),
    instruction="""
You are a parallel coordinator.

FINAL JUDGMENT GUIDANCE:
- Based on your research and results from the agents, output the final results.

FINAL OUTPUT FORMAT (STRICT):
Return ONLY this Markdown template.

## 🧠 Agent Analysis Summary

### 🔍 Labels
| Signal | Label |
|---|---|
| News Coverage | <label> |
| Intent | <label> |
| Sensationalism | <label> |
| Stance | <label> |
| Title vs Body | <label> |
| Context Veracity | <label> |
""",
    tools=[
        AgentTool(news_coverage_agent),
        AgentTool(intent_agent),
        AgentTool(sensationalism_agent),
        AgentTool(stance_agent),
        AgentTool(title_body_agent),
        AgentTool(context_agent),
    ],
)
print("Coordinator defined.")

Coordinator defined.


In [6]:
from google.adk.runners import InMemoryRunner

runner = InMemoryRunner(agent=root_agent)

In [7]:
import json
import asyncio

def load_test_articles(path = os.path.join(parent_dir, 'gen_data/test_article_no_label.json')):
    if not os.path.exists(path):
        print(f"Error: File not found at {path}")
        return []
    
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    return data.get("articles", [])

async def process_batch(articles_batch, batch_name="Batch"):
    print(f"=== Processing {batch_name} ({len(articles_batch)} articles) ===")
    
    for i, art in enumerate(articles_batch):
        headline = art.get('headline', 'No Title')
        print(f"\n[{batch_name}] Article {i+1}: {headline}")
        
        # Prepare Prompt
        prompt = (
            f"Headline: {headline}\n"
            f"Source: {art.get('news_source', 'Unknown')}\n"
            f"Author: {art.get('author', 'Unknown')}\n"
            f"Date: {art.get('date', 'Unknown')}\n\n"
            f"Body:\n{art.get('text', '')}"
        )
        
        # Run Agent using run_debug
        try:
            response = await runner.run_debug(prompt)
            
            # Access output based on ADK response structure
            # Assuming response has .output or similar
            if hasattr(response, 'output'):
                print(response.output)
            else:
                print(response)
                
        except Exception as e:
            print(f"Error running agent: {e}")

        print("-" * 50)
        
        # Sleep slightly to help with rate limits even within batch
        await asyncio.sleep(2)

# Load all data
all_articles = load_test_articles()
print(f"Total articles loaded: {len(all_articles)}")

Total articles loaded: 20


In [8]:
await process_batch(all_articles[0:1], "Batch 1")

=== Processing Batch 1 (1 articles) ===

[Batch 1] Article 1: Trump’s push to end Ukraine war raises fears of 'ugly deal' for Europe

 ### Created new session: debug_session_id

User > Headline: Trump’s push to end Ukraine war raises fears of 'ugly deal' for Europe
Source: Reuters
Author: Andrew Gray
Date: 12/02/2025

Body:
BRUSSELS, Dec 2 (Reuters) - However Donald Trump’s latest push to end the war in Ukraine pans out, Europe fears the prospect of a deal – sooner or later – that will not punish or weaken Russia as its leaders had hoped, placing the continent’s security in greater jeopardy.
Europe may well even have to accept a growing economic partnership between Washington, its traditional protector in the NATO alliance, and Moscow, which most European governments – and NATO itself – say is the greatest threat to European security.
Although Ukrainians and other Europeans managed to push back against parts of a 28-point U.S. plan to end the fighting that was seen as heavily pro-Russi

ParallelCoordinator > ## 🧠 Agent Analysis Summary

### 🔍 Labels
| Signal | Label |
|---|---|
| News Coverage | International Relations |
| Intent | Informative |
| Sensationalism | Neutral |
| Stance | Neutral |
| Title vs Body | Clickbait-free |
| Context Veracity | Inaccurate |
[Event(model_version='gemini-3-flash-preview', content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'request': """Headline: Trump’s push to end Ukraine war raises fears of 'ugly deal' for Europe
Source: Reuters
Author: Andrew Gray
Date: 12/02/2025

Body:
BRUSSELS, Dec 2 (Reuters) - However Donald Trump’s latest push to end the war in Ukraine pans out, Europe fears the prospect of a deal – sooner or later – that will not punish or weaken Russia as its leaders had hoped, placing the continent’s security in greater jeopardy.
Europe may well even have to accept a growing economic partnership between Washington, its traditional protector in the NATO alliance, and Moscow, w

In [9]:
await process_batch(all_articles[1:5], "Batch 2")

=== Processing Batch 2 (4 articles) ===

[Batch 2] Article 1: Hong Kong orders judge-led probe into fire that killed 151

 ### Continue session: debug_session_id

User > Headline: Hong Kong orders judge-led probe into fire that killed 151
Source: Reuters
Author: Clare Jim, Donny Kwok
Date: 12/02/2025

Body:
HONG KONG, Dec 2 (Reuters) - Hong Kong's leader said on Tuesday a judge-led committee will investigate the cause of the city's deadliest fire in decades and review government oversight of renovation practices linked to the blaze that killed at least 151 people.
Police have arrested 13 people for suspected manslaughter, and 12 others in a related corruption probe. Officials said substandard plastic mesh and insulation foam used during renovation works fueled the rapid spread of the fire across seven high-rise towers.
Authorities said they aim to avoid similar tragedies by examining how the fire spread so quickly and the oversight failures around building renovations.

SEARCH AND INVE

In [10]:
await process_batch(all_articles[5:10], "Batch 3")

=== Processing Batch 3 (5 articles) ===

[Batch 3] Article 1: Welcome to fandom’s AI clout economy

 ### Continue session: debug_session_id

User > Headline: Welcome to fandom’s AI clout economy
Source: The Verge
Author: Kat Tenbarge
Date: 12/01/2025

Body:
AI-generated media has become a growing force within online fandom culture, granting fans new ways to manipulate and interact with celebrity likenesses—often without consent and often in controversial ways.
The article opens with a dispute among Ariana Grande fans: one fan account posted AI-edited images of Grande despite her public disapproval of AI recreations of her image. Conflicts like this highlight how fans balance admiration, entitlement, and attention-seeking behaviors.

FANDOM INCENTIVES
Platforms like X (formerly Twitter) now financially reward verified users for engagement, incentivizing "ragebait"—content designed to provoke strong reactions. AI deepfakes and edits fit this perfectly, generating instant attention and of

In [11]:
await process_batch(all_articles[10:15], "Batch 1")

=== Processing Batch 1 (5 articles) ===

[Batch 1] Article 1: Caitlyn Jenner backs IOC move to ban transgender women from Olympics after review finds unfair advantage

 ### Continue session: debug_session_id

User > Headline: Caitlyn Jenner backs IOC move to ban transgender women from Olympics after review finds unfair advantage
Source: Fox News
Author: Madison Colombo
Date: 11/11/2025

Body:
The International Olympic Committee is preparing to ban transgender women from female Olympic events following a scientific review concluding that biological males retain permanent physical advantages.
Former Olympic champion Caitlyn Jenner, a transgender woman, voiced support for restrictions, stating that biological males possess athletic advantages that hormone therapy cannot entirely eliminate.

JENNER'S POSITION
“I am a trans woman, but I am still biologically male,” Jenner said on 'America Reports.' She argued the policy is necessary to protect fairness in women’s sports.
Jenner noted that f

In [12]:
await process_batch(all_articles[15:], "Batch 4")

=== Processing Batch 4 (5 articles) ===

[Batch 4] Article 1: You won’t believe what degrading practice the pope just condemned

 ### Continue session: debug_session_id

User > Headline: You won’t believe what degrading practice the pope just condemned
Source: The Guardian
Author: Unknown
Date: 10/09/2025

Body:
Pope Leo XIV condemned the practice of clickbait as “degrading” during a private audience with international newswire journalists. About 150 members of the global alliance Minds International attended the meeting.

POPE’S MESSAGE ON JOURNALISM
The pope urged journalists to resist sensationalism and prioritize communication as a public good. He argued that clickbait undermines transparency and objectivity.
He praised journalists reporting from conflict zones such as Gaza and Ukraine.

AI AND MEDIA
The pope warned about challenges posed by artificial intelligence, calling for oversight to prevent technological control by a small number of entities.
Key values he emphasized: trans